# CourtIQ court-keypoint training (Colab GPU version)

Same training as `train_court_keypoints.py` in the repo, but run here on a free Colab GPU (NVIDIA T4) instead of local Apple Silicon (MPS), which should be dramatically faster.

**Before running:** In the Colab menu, go to **Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4)**, then click Save. Do this BEFORE running any cells below, or you'll be training on CPU by accident.

You'll need your Roboflow API key (Roboflow -> Settings -> API Keys). It's entered via a hidden prompt in this notebook (not typed into a cell), so it never gets saved into the notebook file itself.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU -- go set Runtime > Change runtime type > GPU before continuing!")

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from getpass import getpass
ROBOFLOW_API_KEY = getpass("Paste your Roboflow API key (input is hidden): ")

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("fyp-3bwmg").project("reloc2-den7l")
dataset = project.version(1).download("yolov8")
print("Downloaded to:", dataset.location)

In [ ]:
from pathlib import Path
from ultralytics import YOLO

# Same config as the local script's round-3 attempt (yolov8s-pose, imgsz=960,
# 200 epochs, mosaic disabled), but a real GPU makes batch=16 (Colab T4 has
# 16GB VRAM) reasonable instead of the CPU-safe batch=8 used locally.
MODEL = "yolov8s-pose.pt"
EPOCHS = 200
IMGSZ = 960
BATCH = 16
MOSAIC = 0.0

data_yaml = Path(dataset.location) / "data.yaml"
assert data_yaml.exists(), f"Expected {data_yaml} to exist -- dataset download may have failed."

model = YOLO(MODEL)
results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    mosaic=MOSAIC,
    task="pose",
    plots=True,
    device=0,  # force GPU explicitly
)

## Check the validation results
Look for the per-class validation table this prints (mAP50, mAP50-95, precision, recall) before trusting the result -- same as the local script would show.

In [ ]:
from pathlib import Path

run_dir = Path(results.save_dir)
best_pt = run_dir / "weights" / "best.pt"
print("Trained weights at:", best_pt)
print("Exists:", best_pt.exists())

## Download the trained weights
This downloads `best.pt` to your computer's Downloads folder. Move it into your local `BbalIQ/models/` folder as `court_keypoints.pt` afterward (overwriting the existing one).

In [ ]:
from google.colab import files
files.download(str(best_pt))